# PitchGraph — validate and build the Library on Google Drive

Runs after `01_ingest.ipynb` has built the lake. Two parts:

1. **Validation suite** on the 12 complete league seasons: style re-identification, the matchup experiment, and the player-absence check. Writes `library/validation.json`, which the Library's Method page shows, whatever the results are.
2. **Library build**: one JSON file per competition-season (team pages and every matchup), the style map, and the app, all in `library/` on Drive. Resumable: re-run after a disconnect and finished seasons are skipped.

Nothing is stored on your laptop. Use a high-RAM runtime if one is available; the largest seasons need a few GB.

> Data: StatsBomb Open Data. Credit StatsBomb (and use their logo) in anything published.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
REPO=https://github.com/akabra13/football_graph_predictor.git
if [ -d /content/pitchgraph ]; then git -C /content/pitchgraph pull -q; else git clone -q $REPO /content/pitchgraph; fi
pip install -q -e /content/pitchgraph
git -C /content/pitchgraph log --oneline -1

In [ ]:
import os, json
os.environ['PITCHGRAPH_DATA'] = '/content/drive/MyDrive/pitchgraph'

import polars as pl
from pitchgraph.config import require_data_root
from pitchgraph.data.lake import Lake
from pitchgraph.chains import validate

lake = Lake.open()
library = require_data_root() / 'library'
library.mkdir(parents=True, exist_ok=True)
print('lake matches:', lake.matches().height, '| library:', library)

## 1. Validation suite
Each season is analysed on its own: every prediction is made from other matches only. Results go into the Library as they are.

In [ ]:
def season(key):
    cid, sid = (int(x) for x in key.split('_'))
    events = lake.events().filter((pl.col('competition_id') == cid) & (pl.col('season_id') == sid))
    ids = lake.matches().filter((pl.col('competition_id') == cid) & (pl.col('season_id') == sid))['match_id']
    lineups = lake.scan('lineups').filter(pl.col('match_id').is_in(ids)).collect()
    return events, lineups

results = validate.run_suite([(label, *season(key)) for key, label in validate.COMPLETE_LEAGUES.items()])
(library / 'validation.json').write_text(json.dumps(results, indent=1, default=str))
print(json.dumps({k: results[k] for k in ('missed', 'scope') if k in results}, indent=1))

In [ ]:
rows = [{'league': r[0], 'style re-ID': round(r[1], 2), 'chance': round(r[2], 2),
         'lanes: matchup vs own (90% CI)': f'{l[1]:+.4f} to {l[2]:+.4f}',
         'volume: matchup vs strength (90% CI)': f'{v[1]:+.4f} to {v[2]:+.4f}'}
        for r, l, v in zip(results['reid'], results['lanes'], results['volume'])]
pl.Config.set_tbl_rows(20); pl.Config.set_tbl_width_chars(200)
pl.DataFrame(rows)

## 2. Build the Library
Fits the value model on every competition, then writes each competition-season's team pages and matchups. Safe to re-run.

In [ ]:
from pitchgraph.library.build import build_all
from pitchgraph.library.package import copy_app

index = build_all(lake, library)
copy_app(library)
print(len(index['seasons']), 'competition-seasons,',
      sum(len(s['teams']) for s in index['seasons'].values()), 'team pages')

## 3. Publish
The Library is a folder on Drive (`pitchgraph/library`). To publish it as a private page, zip it here, then ask Claude Code to publish `library.zip`: it is unpacked in a temporary folder, published, and deleted, so it never stays on your laptop.

In [ ]:
import shutil
archive = shutil.make_archive(str(require_data_root() / 'library'), 'zip', library)
size = sum(p.stat().st_size for p in library.rglob('*') if p.is_file() and p.suffix != '.npz')
print(archive, '| published size %.1f MB' % (size / 1e6))